# Simple acquisition on the Ti2-E

A minimal real-time experiment on the Ti2-E: one field of view, imaged
every few seconds. Every frame is segmented with Cellpose, tracked
and measured while the next one is acquired. There is no stimulation.

napari with napari-micromanager gives live view and stage control, so
you can find the field before the run and watch the frames come in
during it.

Put a sample with a nuclear marker (for example H2B-miRFP) on the stage.

## 1. Connect to the Ti2-E

`Ti2E()` loads `NikonTi2e.cfg`, applies the `System/Startup` preset and
selects the `Channel` group. Pass `config=r"..."` to load another file.
Close NIS-Elements and any other Micro-Manager first: a device can only be
opened by one program at a time.

In [ ]:
from faro_aoki.ti2e import Ti2E
import faro.core.utils as utils

mic = Ti2E()
utils.print_configs(mic.mmc)

## 2. Open napari

napari-micromanager runs on the core of `mic`. Use **Live** to focus and
the stage controls to move to the field you want; the experiment below
images the stage position at the time you build it.

Do not run `faro-napari ti2e` at the same time: it opens the hardware a
second time.

In [ ]:
import napari
from napari_micromanager import MainWindow

viewer = napari.Viewer()
mm_widget = MainWindow(viewer, mmcore=mic.mmc)
# No name=: a named dock of the napari-micromanager window crashes.
viewer.window.add_dock_widget(mm_widget)

## 3. Settings

`CHANNEL` must be a preset of the `Channel` group listed above.
`SEGMENTER` picks the segmentation: `"cellpose"` (Cellpose-SAM, needs
`uv sync --group cellpose` and a GPU) or `"otsu"` (a threshold, no extra
install, for a bright nuclear marker on a dark background).
`WRITER` picks the storage format: `"omezarr"` writes one OME-Zarr
store, `"tiff"` one TIFF file per frame. Every run gets its own
folder in `data/` next to this notebook, so nothing is overwritten.

In [ ]:
import os
from datetime import datetime

CHANNEL = {"config": "iRFP", "exposure": 100}  # nuclear marker
INTERVAL_S = 5.0  # seconds between frames
N_FRAMES = 10
SEGMENTER = "cellpose"  # or "otsu"
WRITER = "omezarr"  # or "tiff"

path = os.path.abspath(
    os.path.join("data", f"simple-acquisition-{datetime.now():%Y%m%d-%H%M%S}")
)  # data/ next to this notebook

## 4. Set up the processing pipeline

Cellpose (or Otsu thresholding) for segmentation, trackpy for tracking
and `SimpleFE` for area and intensity per cell.

Cellpose runs on the GPU only if torch sees CUDA. Without it, Cellpose
falls back to the CPU without an error, where it is very slow and can
run out of memory on a full camera frame (the kernel then dies without a
message). The torch build of this environment (`+cu130`) needs an NVIDIA
driver >= 580.65; `nvidia-smi` shows the installed version.

In [ ]:
from faro.core.data_structures import SegmentationMethod
from faro.core.pipeline import ImageProcessingPipeline
from faro.feature_extraction.simple import SimpleFE
from faro.tracking.trackpy import TrackerTrackpy

if SEGMENTER == "cellpose":
    import torch
    from faro.segmentation.cellpose_v4 import CellposeV4

    if torch.cuda.is_available():
        print(
            f"Cellpose on {torch.cuda.get_device_name(0)} (torch {torch.__version__})"
        )
    else:
        print(
            f"WARNING: torch {torch.__version__} sees no CUDA GPU, Cellpose runs on "
            "the CPU. Update the NVIDIA driver or set SEGMENTER = 'otsu'."
        )
    segmentator = CellposeV4()
else:
    from faro.segmentation.base import OtsuSegmentator

    segmentator = OtsuSegmentator()

segmentators = [
    SegmentationMethod(
        name="labels",
        segmentation_class=segmentator,
        use_channel=0,
        save_tracked=True,
    )
]

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=SimpleFE("labels"),
    tracker=TrackerTrackpy(),
)

## 5. Preview segmentation

Snap one frame at the current position and segment it. The image and
the labels open as layers in napari. Adjust the exposure or focus until
the labels look right. Stop **Live** first.

In [ ]:
mic.mmc.setConfig(mic.CHANNEL_GROUP, CHANNEL["config"])
mic.mmc.setExposure(CHANNEL["exposure"])
mic.mmc.snapImage()
test_img = mic.mmc.getImage()
labels = segmentators[0].segmentation_class.segment(test_img)

viewer.add_image(test_img, name=f"preview {CHANNEL['config']}")
viewer.add_labels(labels, name="preview labels")
print(f"{labels.max()} objects")

## 6. Define the experiment

The position is the current stage position. It has no `z`, so faro
never moves the focus drive: the focus stays where it is, or where
the PFS holds it.

In [ ]:
from faro.core.data_structures import RTMSequence

x0, y0 = mic.mmc.getXYPosition()
seq = RTMSequence(
    time_plan={"interval": INTERVAL_S, "loops": N_FRAMES},
    stage_positions=[{"x": x0, "y": y0}],
    channels=[CHANNEL],
)

events = list(seq)
print(f"{len(events)} events at x={x0:.1f} um, y={y0:.1f} um")
events[0]

## 7. Run the experiment

`validate_events` checks the channel names and exposures against the
hardware before anything moves. `run_experiment` returns at once: the
acquisition and the pipeline run in the background, and napari stays
responsive and shows the frames. The experiment status dock opens
with the run.

In [ ]:
from faro.core.controller import Controller
from faro.core.writers import OmeZarrWriter, TiffWriter
from faro.widgets import ExperimentStatusWidget

if WRITER == "omezarr":
    writer = OmeZarrWriter(storage_path=path)
else:
    writer = TiffWriter(storage_path=path)

ctrl = Controller(mic, pipeline, writer=writer)
assert ctrl.validate_events(events), "fix the warnings above before running"
# Experiment status (progress, lag, errors), docked when the run starts.
viewer.window.add_dock_widget(
    ExperimentStatusWidget(ctrl), name="experiment status", area="right"
)
handle = ctrl.run_experiment(events)

`finish_experiment()` waits for the run, drains the pipeline and closes
the writer. napari does not update while this cell runs, so run it when
the status dock shows the run is done.

In [ ]:
handle.wait()
ctrl.finish_experiment()
print("results in", path)

## 8. Inspect the results

Tracks are one parquet file per field of view. The images are in
`acquisition.ome.zarr` (OME-Zarr) or in `raw/` and `labels/`, one
file per frame (TIFF).

In [ ]:
import pandas as pd

tracks = pd.read_parquet(os.path.join(path, "tracks", "0_latest.parquet"))
print(
    f"Tracked {tracks['particle'].nunique()} cells across "
    f"{tracks['timestep'].nunique()} frames"
)
tracks.head()

In [ ]:
import matplotlib.pyplot as plt
from glob import glob

import numpy as np
import tifffile
import zarr


def first_channel(frame):
    """First channel of a (y, x) or (c, y, x) frame."""
    frame = np.asarray(frame)
    return frame if frame.ndim == 2 else frame[0]


if WRITER == "omezarr":
    store = zarr.open(os.path.join(path, "acquisition.ome.zarr"), mode="r")
    raw = store["0"]  # (t, c, y, x) for one position
    lbl = store["labels/labels/0"]  # (t, y, x)
    first, last, first_lbl = raw[0], raw[-1], lbl[0]
else:
    raw_files = sorted(glob(os.path.join(path, "raw", "*.tiff")))
    lbl_files = sorted(glob(os.path.join(path, "labels", "*.tiff")))
    first, last = tifffile.imread(raw_files[0]), tifffile.imread(raw_files[-1])
    first_lbl = tifffile.imread(lbl_files[0])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(first_channel(first), cmap="gray")
axes[0].set_title("Raw (first frame)")
axes[1].imshow(first_channel(last), cmap="gray")
axes[1].set_title("Raw (last frame)")
axes[2].imshow(first_lbl, cmap="nipy_spectral")
axes[2].set_title("Labels (first frame)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()